## 文本分类-BERT + emotion数据集 + 微调：获取嵌入向量

In [1]:
import torch
from datasets import load_dataset
from transformers import BertModel, BertTokenizer

### 1. 加载数据并提取嵌入向量

In [2]:
# 获取数据集
ds = load_dataset("emotion")

# 加载预训练模型
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# 模型加载到GPU
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# 分词
def tokenize_handle(batch):
    # 通过传递给ds对象的map方法，会给ds添加3列：['input_ids', 'token_type_ids', 'attention_mask']
    return tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt")

# 嵌入
def embedding_handle(batch):
    # 模型所需的输入数据
    inputs = {
        k: v.to(device) for k, v in batch.items()
        if k in tokenizer.model_input_names
    }

    # 调用模型
    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = outputs["last_hidden_state"]
        
    # 文本分类任务，所以只需要返回第一个[CLS]的特征值即可
    return {"hidden_state": last_hidden_state[:,0].cpu()}

# 给数据集分词
ds = ds.map(tokenize_handle, batched=True, batch_size=1000)

# 给数据集设置为张量
ds.set_format("torch", columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])

# 给数据集添加隐藏状态[CLS]的特征值
ds = ds.map(embedding_handle, batch_size=1000, batched=True)

Using the latest cached version of the module from /Users/alex.zhou/.cache/huggingface/modules/datasets_modules/datasets/emotion/cca5efe2dfeb58c1d098e0f9eeb200e9927d889b5a03c67097275dfb5fe463bd (last modified on Fri May 31 14:39:28 2024) since it couldn't be found locally at emotion, or remotely on the Hugging Face Hub.


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 2000
    })
})

In [4]:
type(ds)

datasets.dataset_dict.DatasetDict

### 2. 保存数据集

In [5]:
ds_file_path = "../../../data/datasets/emotion/embedding_ds"

In [6]:
!ls {ds_file_path}

ls: ../../../data/datasets/emotion/embedding_ds: No such file or directory


In [7]:
ds.save_to_disk(ds_file_path)

Saving the dataset (0/1 shards):   0%|          | 0/16000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
# 再次目录文件列表
!ls {ds_file_path}

dataset_dict.json test              train             validation


### 3. 从磁盘中加载数据集

In [9]:
from datasets import load_from_disk

In [10]:
ds2 = load_from_disk(ds_file_path)
ds2

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'],
        num_rows: 2000
    })
})

In [11]:
type(ds2["train"][0])

dict

In [12]:
ds2["train"][0].keys()

dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask', 'hidden_state'])

In [13]:
# 设置一下数据集的格式，方便查看
ds2.set_format("pandas")

In [14]:
train_df = ds2["train"][:]

In [15]:
# 查看头部的几行数据
train_df.head()

,text,label,input_ids,token_type_ids,attention_mask,hidden_state
0,i didnt feel humiliated,0,"[101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.08821961, 0.3044226, -0.23050076, -0.07414..."
1,i can go from feeling so hopeless to so damned...,0,"[101, 1045, 2064, 2175, 2013, 3110, 2061, 2062...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.25537017, -0.008101954, -0.0044731945, -0.3..."
2,im grabbing a minute to post i feel greedy wrong,3,"[101, 10047, 9775, 1037, 3371, 2000, 2695, 104...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...","[0.14182423, 0.53024393, 0.3241856, -0.3340335..."
3,i am ever feeling nostalgic about the fireplac...,2,"[101, 1045, 2572, 2412, 3110, 16839, 9080, 128...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-0.2051316, 0.3283987, 0.3943012, -0.4248372,..."
4,i am feeling grouchy,3,"[101, 1045, 2572, 3110, 24665, 7140, 11714, 10...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0.19088942, 0.592778, 0.16961585, -0.47933453..."


**后续我们需要使用emotions的数据集，就可以直接从磁盘中加载了，而不需要每次都下载数据集，然后计算嵌入向量。**